# Keepa Pipeline — Data Explorer

Browse every table in the Amazon Tracker database with sample queries.

**Tables (jump to a section):**
1. [`dim_product`](#1-dim_product) — one row per ASIN
2. [`fct_keepa_daily`](#2-fct_keepa_daily) — one row per (ASIN, day) — Keepa Viewer CSV import
3. [`v_daily_sales`](#3-v_daily_sales) — derived daily sales from FBA stock deltas
4. [`fct_keepa_seller_history`](#4-fct_keepa_seller_history) — per-seller stock & price events from API
5. [`dim_keepa_seller`](#5-dim_keepa_seller) — seller dimension
6. [`asin_api_state`](#6-asin_api_state) — API fetch queue state
7. [`fct_asin_daily`](#7-fct_asin_daily) — AOD scraper output (sellers JSON)
8. [Cross-table analyses](#8-cross-table-analyses) — joins, top movers, recommendations


## Setup

In [ ]:
import sqlite3
import pandas as pd
import plotly.express as px

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 80)

DB = '../pipeline/amazon_tracker.db'
conn = sqlite3.connect(DB)

def q(sql, **params):
    return pd.read_sql_query(sql, conn, params=params)

# Catalog summary
tables = q("SELECT name FROM sqlite_master WHERE type IN ('table','view') ORDER BY name")
print(f"Database: {DB}")
print(f"Tables & views: {len(tables)}")
tables

## 1. `dim_product`
One row per ASIN being tracked. Slowly-changing dimension with title, brand, parent ASIN, color/size variation, and image URL.

In [ ]:
# Schema
q("PRAGMA table_info(dim_product)")

In [ ]:
# Sample rows + counts
print(f"Total products: {q('SELECT COUNT(*) AS n FROM dim_product').iloc[0]['n']}")
print(f"  with title:    {q(\"SELECT COUNT(*) AS n FROM dim_product WHERE title IS NOT NULL\").iloc[0]['n']}")
print(f"  with image:    {q(\"SELECT COUNT(*) AS n FROM dim_product WHERE image_url IS NOT NULL\").iloc[0]['n']}")
print(f"  unique brands: {q('SELECT COUNT(DISTINCT brand) AS n FROM dim_product').iloc[0]['n']}")
q("SELECT asin, brand, variation_size, variation_color, substr(title,1,50) AS title FROM dim_product LIMIT 10")

In [ ]:
# Top brands by ASIN count
q("SELECT brand, COUNT(*) AS asins FROM dim_product GROUP BY brand ORDER BY asins DESC LIMIT 15")

## 2. `fct_keepa_daily`
**One row per (ASIN, snapshot_date)** — populated by `keepa_viewer_export.py`.

Holds the most-used daily Keepa fields as **typed columns**, plus the full original CSV row as `raw_json` so nothing is lost. Cumulative (never purged).

In [ ]:
# Schema
q("PRAGMA table_info(fct_keepa_daily)")

In [ ]:
# Coverage per day
q("""
SELECT snapshot_date,
       COUNT(*)                                              AS asins,
       SUM(CASE WHEN buy_box_price IS NOT NULL THEN 1 END)   AS with_buy_box,
       SUM(CASE WHEN fba_stock     IS NOT NULL THEN 1 END)   AS with_fba_stock,
       SUM(CASE WHEN monthly_sold_num IS NOT NULL THEN 1 END) AS with_monthly_sold
FROM fct_keepa_daily
GROUP BY snapshot_date ORDER BY snapshot_date
""")

In [ ]:
# Latest day — top sellers by FBA stock
q("""
SELECT k.asin, substr(d.title,1,55) AS title,
       k.sales_rank_current AS bsr, k.buy_box_price AS price,
       k.fba_stock, k.fba_offers, k.fbm_offers, k.total_offers,
       k.buy_box_seller
FROM fct_keepa_daily k LEFT JOIN dim_product d ON d.asin=k.asin
WHERE k.snapshot_date = (SELECT MAX(snapshot_date) FROM fct_keepa_daily)
  AND k.fba_stock IS NOT NULL
ORDER BY k.fba_stock DESC LIMIT 15
""")

In [ ]:
# Day-over-day changes for ONE ASIN
q("""
SELECT snapshot_date, sales_rank_current AS bsr, buy_box_price, buy_box_stock,
       fba_stock, fba_offers, fbm_offers, total_offers, oos_90d_pct,
       pct_top_seller_30d, pct_top_seller_90d, monthly_sold_num
FROM fct_keepa_daily
WHERE asin = 'B0FVTS2NJ5'
ORDER BY snapshot_date
""")

In [ ]:
# Peek the raw CSV row Keepa exported — every field even those not promoted to typed columns
import json
row = q("SELECT raw_json FROM fct_keepa_daily WHERE asin='B0FVTS2NJ5' LIMIT 1").iloc[0]['raw_json']
for k, v in json.loads(row).items():
    print(f"  {k:<55} = {v!r}")

## 3. `v_daily_sales` (view)
Derives daily units sold per ASIN from `fba_stock` deltas in `fct_keepa_daily`.

- When stock goes **down**, units_sold = (prev - curr)
- When stock goes **up**, restock detected → units_sold = NULL (we can't measure)
- `units_sold_per_day` normalizes by days between snapshots

**Limitation:** aggregate only — restocks hide some sales. The per-seller version (#4) is more accurate.

In [ ]:
# Top movers (latest day, by derived velocity)
q("""
SELECT v.asin, substr(d.title,1,55) AS title,
       v.fba_stock_prev AS prev, v.fba_stock AS curr, v.days_since_prev,
       v.units_sold, ROUND(v.units_sold_per_day, 2) AS units_per_day
FROM v_daily_sales v LEFT JOIN dim_product d ON d.asin = v.asin
WHERE v.snapshot_date = (SELECT MAX(snapshot_date) FROM v_daily_sales)
  AND v.units_sold_per_day > 0
ORDER BY v.units_sold_per_day DESC LIMIT 20
""")

## 4. `fct_keepa_seller_history`
**Per-seller stock & price change events** — populated by `keepa_api_offers.py --tick`.

One row per change event observed by Keepa. From this we can reconstruct every seller's stock at any past point.

*Will be empty until the API collector has run a few ticks.*

In [ ]:
# Schema
q("PRAGMA table_info(fct_keepa_seller_history)")

In [ ]:
# Top-line stats
total = q("SELECT COUNT(*) AS n FROM fct_keepa_seller_history").iloc[0]['n']
stk   = q("SELECT COUNT(*) AS n FROM fct_keepa_seller_history WHERE stock IS NOT NULL").iloc[0]['n']
prc   = q("SELECT COUNT(*) AS n FROM fct_keepa_seller_history WHERE price_cents IS NOT NULL").iloc[0]['n']
asins = q("SELECT COUNT(DISTINCT asin) AS n FROM fct_keepa_seller_history").iloc[0]['n']
sellers = q("SELECT COUNT(DISTINCT seller_id) AS n FROM fct_keepa_seller_history").iloc[0]['n']
print(f"Total change events: {total:,}")
print(f"  stock events:     {stk:,}")
print(f"  price events:     {prc:,}")
print(f"  distinct ASINs:   {asins:,}")
print(f"  distinct sellers: {sellers:,}")

In [ ]:
# Latest stock per (asin, seller) — derived from most recent change event
q("""
WITH latest AS (
    SELECT asin, seller_id, MAX(change_time) AS last_change
    FROM fct_keepa_seller_history
    WHERE stock IS NOT NULL
    GROUP BY asin, seller_id
)
SELECT h.asin, h.seller_id, s.seller_name, h.is_fba, h.is_prime,
       h.stock, h.change_time
FROM fct_keepa_seller_history h
JOIN latest l ON l.asin = h.asin AND l.seller_id = h.seller_id AND l.last_change = h.change_time
LEFT JOIN dim_keepa_seller s ON s.seller_id = h.seller_id
WHERE h.asin = 'B0FVTS2NJ5'
ORDER BY h.stock DESC NULLS LAST
""")

In [ ]:
# Stock history for ONE seller on ONE ASIN — see every change
q("""
SELECT change_time, stock, price_cents/100.0 AS price_usd
FROM fct_keepa_seller_history
WHERE asin='B0FVTS2NJ5' AND seller_id='ALEHKFC9FPA50'
  AND stock IS NOT NULL
ORDER BY change_time
""")

## 5. `dim_keepa_seller`
Seller dimension. Keepa-stable merchant IDs (e.g. `ALEHKFC9FPA50`) → name, rating, review count.

In [ ]:
q("SELECT * FROM dim_keepa_seller ORDER BY updated_at DESC LIMIT 15")

## 6. `asin_api_state`
Tracks the API fetch queue. `last_fetched_at` IS NULL = never fetched (top priority next tick).

In [ ]:
q("""
SELECT
  COUNT(*)                                            AS total,
  SUM(CASE WHEN last_fetched_at IS NULL THEN 1 END)   AS never_fetched,
  SUM(CASE WHEN fetch_success = 1 THEN 1 END)         AS successful_fetches,
  SUM(CASE WHEN fetch_success = 0 THEN 1 END)         AS failed_fetches,
  MIN(last_fetched_at)                                AS oldest_fetch,
  MAX(last_fetched_at)                                AS newest_fetch
FROM asin_api_state
""")

## 7. `fct_asin_daily`
Output of the AOD scraper (`browser_collector.py`). One row per (ASIN, day) with `sellers` as JSON array.

In [ ]:
q("""
SELECT snapshot_date, asin, bsr_rank, total_sellers, fba_sellers, fbm_sellers,
       min_price, max_price, units_sold, fba_units_sold, fbm_units_sold
FROM fct_asin_daily
ORDER BY snapshot_date DESC LIMIT 10
""")

## 8. Cross-table analyses

In [ ]:
# ASINs ranked by recent BSR improvement (lower = better)
q("""
WITH win AS (
    SELECT asin, MIN(snapshot_date) AS d_first, MAX(snapshot_date) AS d_last
    FROM fct_keepa_daily GROUP BY asin
    HAVING COUNT(DISTINCT snapshot_date) >= 2
)
SELECT w.asin, substr(d.title,1,55) AS title,
       first.sales_rank_current AS bsr_first, last.sales_rank_current AS bsr_last,
       last.sales_rank_current - first.sales_rank_current AS bsr_change
FROM win w
JOIN fct_keepa_daily first ON first.asin = w.asin AND first.snapshot_date = w.d_first
JOIN fct_keepa_daily last  ON last.asin  = w.asin AND last.snapshot_date  = w.d_last
LEFT JOIN dim_product d ON d.asin = w.asin
WHERE first.sales_rank_current IS NOT NULL AND last.sales_rank_current IS NOT NULL
ORDER BY bsr_change ASC LIMIT 15
""")

In [ ]:
# Buy-box dominance landscape — who controls the most ASINs
q("""
SELECT buy_box_seller, COUNT(*) AS asins_won_today,
       AVG(buy_box_price) AS avg_price, AVG(pct_top_seller_30d) AS avg_30d_dominance
FROM fct_keepa_daily
WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM fct_keepa_daily)
  AND buy_box_seller IS NOT NULL AND buy_box_seller != '-'
GROUP BY buy_box_seller
ORDER BY asins_won_today DESC LIMIT 15
""")

In [ ]:
# BSR distribution chart (latest day)
df = q("""SELECT sales_rank_current FROM fct_keepa_daily
          WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM fct_keepa_daily)
            AND sales_rank_current IS NOT NULL""")
px.histogram(df, x='sales_rank_current', nbins=40, title='BSR distribution today',
             labels={'sales_rank_current':'Best Sellers Rank'}).show()

In [ ]:
# Inventory + sellers landscape (totals across catalog)
q("""
SELECT snapshot_date,
       SUM(fba_stock)    AS total_fba_stock,
       SUM(buy_box_stock) AS total_bb_stock,
       SUM(fba_offers)   AS total_fba_sellers,
       SUM(fbm_offers)   AS total_fbm_sellers,
       AVG(oos_90d_pct)  AS avg_90d_oos_pct
FROM fct_keepa_daily
GROUP BY snapshot_date ORDER BY snapshot_date
""")

## Tips
- Add your own query cells anywhere — just `q("""SELECT ...""")` returns a DataFrame.
- For VS Code's built-in SQLite viewer, see `docs/vscode_sqlite_setup.md`.
- All times in `fct_keepa_seller_history` are UTC.
- Price is in cents (`price_cents / 100.0` = dollars).